# Занятие 36. Практика: бустинг для прогноза MMR

**Теория:** `Урок_35_Boosting_Теория/35_Градиентный_бустинг.ipynb` (занятие 35).  
**Задача:** регрессия — предсказать **следующий MMR** игрока (рейтинг навыка).  
**Главная метрика:** **MAE** (mean absolute error — средняя абсолютная ошибка в очках MMR). RMSE считаем рядом для сравнения.  
**Модельный фокус:** внешние бустинги **XGBoost**, **LightGBM**, **CatBoost** (+ baseline и при желании `HistGradientBoosting` из sklearn).

> Этот ноутбук — **авторский вариант с решениями** (для преподавателя). Студентам выдаётся копия без готовых ответов.


## Легенда: продукт RankPulse

Команда киберспортивной платформы **RankPulse** строит подсказку для матчмейкинга: «какой MMR будет у игрока после ближайшего окна матчей?»

Если прогноз уверенный — можно мягко подкрутить подбор соперников. Если ошибка может быть большой — лучше **не трогать** рейтинг автоматически и отдать кейс аналитику.

Признаки (синтетический лог ~800 игроков): винрейт, число матчей, роль, часы в игре, текущий MMR, стрик побед/поражений, средний KDA, доля игр в пати. Цель — `next_mmr`.


## Что нужно сделать

1. Придумать правило: когда доверять автопрогнозу MMR.
2. Собрать и осмотреть синтетический датасет.
3. Построить графики (распределения и сравнение по ролям).
4. Разбить данные на train / validation / test.
5. Поставить baseline (`DummyRegressor`) и зафиксировать **MAE**.
6. Обучить `HistGradientBoostingRegressor` с early stopping (sklearn — для ориентира).
7. Обязательно сравнить **XGBoost**, **LightGBM**, **CatBoost** (с early stopping где уместно).
8. Собрать единый рейтинг моделей и таблицу плюсов/минусов.
9. Выбрать победителя по validation MAE и один раз проверить на test.

### Оценивание (30 баллов)

| № | Тема | Баллы |
|---|------|------:|
| 1 | Мини-симуляция: порог доверия к прогнозу | 2 |
| 2 | Датасет и осмотр | 3 |
| 3 | Диаграммы | 3 |
| 4 | Train / validation / test | 2 |
| 5 | Baseline и метрика MAE | 3 |
| 6 | HistGradientBoosting и early stopping | 3 |
| 7 | Внешние бустинги: XGBoost, LightGBM, CatBoost | 7 |
| 8 | Единый рейтинг моделей | 3 |
| 9 | Плюсы и минусы библиотек | 2 |
| 10 | Финальный вывод и проверка на test | 2 |
| | **Итого** | **30** |


---
## Среда и библиотеки

В учебной среде `xgboost`, `lightgbm` и `catboost` обычно уже установлены. Если импорт падает дома, выполните в терминале:

```bash
pip install xgboost lightgbm catboost
```


In [ ]:
# Дома (если нужно): pip install xgboost lightgbm catboost
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor

RANDOM_STATE = 42
TEST_SIZE = 0.2
VAL_SIZE = 0.25  # 25% от train_val ≈ 20% от всех данных
PRIMARY_METRIC = "MAE"  # главная метрика практики (как в теории занятия 35)
N_PLAYERS = 800


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def evaluate_regression(model_name, y_true, y_pred, fit_time_sec=None):
    row = {
        "model": model_name,
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": rmse(y_true, y_pred),
        "R2": float(r2_score(y_true, y_pred)),
    }
    if fit_time_sec is not None:
        row["fit_time_sec"] = float(fit_time_sec)
    return row


print("PRIMARY_METRIC =", PRIMARY_METRIC)


---
## Задание 1. Мини-симуляция: порог доверия к прогнозу — **2 балла**

Пока модель ещё не обучена, зафиксируйте **бизнес-правило** RankPulse: когда автопрогноз MMR можно применять в матчмейкинге, а когда — отправлять на ручную проверку.

### Что сделать

**Шаг 1.** Создайте словарь `trust_rule` с ключами:
- `max_abs_error_mmr` — максимальная допустимая |ошибка| в очках MMR (число > 0);
- `auto_action` — текст действия при «уверенном» прогнозе;
- `manual_action` — текст действия при риске большой ошибки.

**Шаг 2.** Создайте таблицу `incoming_players` (≥ 4 строки) с колонками:
`player_id`, `predicted_next_mmr`, `abs_error_estimate` (оценка |ошибки| прототипа).

**Шаг 3.** Добавьте колонку `decision`: если `abs_error_estimate <= max_abs_error_mmr` → `auto_action`, иначе → `manual_action`.

**Шаг 4.** В markdown ниже (1–2 предложения) объясните: почему даже при хорошем среднем MAE отдельным игрокам с большой оценкой ошибки нужна ручная проверка.

### Подробные критерии (для проверки LLM)
- **0.5 балла:** в `trust_rule` есть все 3 ключа; `max_abs_error_mmr` > 0.
- **0.5 балла:** `incoming_players` ≥ 4 строк и нужные 3 исходные колонки.
- **0.5 балла:** `decision` заполнена по порогу (`<=` → авто, иначе → ручная проверка).
- **0.5 балла:** есть прикладное объяснение про MAE vs ручную проверку.

### Снижение баллов
- Нет `decision` или она не зависит от порога → не выше **1.0**.
- Нет объяснения в markdown → минус **0.5**.


In [ ]:
trust_rule = {
    "max_abs_error_mmr": 80,
    "auto_action": "применить прогноз в матчмейкинге",
    "manual_action": "отправить аналитику на ручную проверку",
}

incoming_players = pd.DataFrame(
    {
        "player_id": [1001, 1002, 1003, 1004],
        "predicted_next_mmr": [3120, 2450, 4010, 2785],
        "abs_error_estimate": [35, 120, 55, 95],
    }
)

incoming_players["decision"] = np.where(
    incoming_players["abs_error_estimate"] <= trust_rule["max_abs_error_mmr"],
    trust_rule["auto_action"],
    trust_rule["manual_action"],
)

incoming_players


*Почему нужна ручная проверка?*

MAE показывает среднюю ошибку по всем игрокам. У конкретного игрока оценка ошибки может быть намного больше среднего: если автоматически сдвинуть ему подбор соперников, матчи станут нечестными. Поэтому при большой оценке ошибки безопаснее отдать кейс аналитику.


---
## Задание 2. Датасет и осмотр — **3 балла**

Соберите синтетический лог игроков RankPulse (~500–1000 строк) с понятными колонками и целевой переменной `next_mmr`.

### Что сделать

**Шаг 1.** Сгенерируйте DataFrame `players` (рекомендуется `N_PLAYERS = 800`) с колонками не ниже:
`winrate`, `matches_played`, `role`, `hours_played`, `current_mmr`, `streak`, `avg_kda`, `party_rate`, `next_mmr`.

**Шаг 2.** Сделайте one-hot по `role` (или иной численный код), соберите матрицу признаков `X` и цель `y = next_mmr`. В `X` не должно быть `next_mmr`.

**Шаг 3.** Выведите `players.shape`, `players.head()`, базовую статистику числовых колонок и распределение ролей (`value_counts`).

### Подробные критерии (для проверки LLM)
- **1.0 балл:** датасет ≥ 500 строк, есть все ключевые колонки и цель `next_mmr`.
- **1.0 балл:** корректно собраны `X` и `y`; цель не попала в признаки.
- **1.0 балл:** показаны размер, `head`, описание чисел и частоты ролей.

### Снижение баллов
- Меньше 500 строк → минус **1.0**.
- Утечка цели в `X` → минус **1.0**.
- Нет осмотра (`head` / describe / роли) → минус **0.5** за каждый пропуск.


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

roles = rng.choice(
    ["carry", "mid", "support", "offlane"],
    size=N_PLAYERS,
    p=[0.30, 0.25, 0.25, 0.20],
)
role_effect = {"carry": 35.0, "mid": 20.0, "support": -15.0, "offlane": 10.0}

winrate = rng.beta(5.5, 4.5, N_PLAYERS)
matches_played = rng.integers(80, 2200, N_PLAYERS)
hours_played = rng.integers(120, 5200, N_PLAYERS)
current_mmr = rng.integers(1200, 4800, N_PLAYERS).astype(float)
streak = rng.integers(-6, 12, N_PLAYERS)
avg_kda = rng.uniform(1.2, 6.5, N_PLAYERS)
party_rate = rng.uniform(0.0, 1.0, N_PLAYERS)

noise = rng.normal(0.0, 55.0, N_PLAYERS)
next_mmr = (
    current_mmr
    + 220.0 * (winrate - 0.5)
    + 0.015 * matches_played
    + 0.004 * hours_played
    + 9.0 * streak
    + 18.0 * avg_kda
    - 25.0 * party_rate
    + np.array([role_effect[r] for r in roles])
    + noise
)

players = pd.DataFrame(
    {
        "winrate": winrate,
        "matches_played": matches_played,
        "role": roles,
        "hours_played": hours_played,
        "current_mmr": current_mmr,
        "streak": streak,
        "avg_kda": avg_kda,
        "party_rate": party_rate,
        "next_mmr": next_mmr,
    }
)

# Числовые признаки + one-hot по роли (удобно для всех моделей одинаково)
X = pd.get_dummies(players.drop(columns=["next_mmr"]), columns=["role"], dtype=float)
y = players["next_mmr"].copy()

print("players:", players.shape)
print("X:", X.shape, "y:", y.shape)
print("\nРоли:")
print(players["role"].value_counts())
print("\nПервые строки:")
print(players.head())
players.describe().T


---
## Задание 3. Диаграммы — **3 балла**

Покажите данные глазами: распределение цели и связь ключевых признаков с MMR.

### Что сделать

**Шаг 1.** Гистограмма `next_mmr` (заголовок + подпись оси X).

**Шаг 2.** Boxplot `next_mmr` по `role` (заголовок + подписи осей).

**Шаг 3.** Scatter: `current_mmr` vs `next_mmr` **или** bar: средний `next_mmr` по бинам винрейта — на выбор, но с заголовком и подписями осей.

### Подробные критерии (для проверки LLM)
- **1.0 балл:** гистограмма `next_mmr` с заголовком и подписью оси.
- **1.0 балл:** boxplot по ролям с заголовком и подписями осей.
- **1.0 балл:** третий график (scatter или bar) с заголовком и подписями осей.

### Снижение баллов
- График без заголовка или без подписей осей → минус **0.5** за каждый такой график.
- Нет одного из трёх требуемых типов визуализации → минус **1.0**.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(players["next_mmr"], bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("Распределение следующего MMR")
axes[0].set_xlabel("next_mmr")
axes[0].set_ylabel("Число игроков")

roles_order = ["carry", "mid", "offlane", "support"]
box_data = [players.loc[players["role"] == r, "next_mmr"] for r in roles_order]
axes[1].boxplot(box_data, tick_labels=roles_order, showfliers=False)
axes[1].set_title("next_mmr по ролям")
axes[1].set_xlabel("role")
axes[1].set_ylabel("next_mmr")

axes[2].scatter(
    players["current_mmr"],
    players["next_mmr"],
    alpha=0.35,
    s=18,
    color="darkorange",
)
axes[2].set_title("Текущий vs следующий MMR")
axes[2].set_xlabel("current_mmr")
axes[2].set_ylabel("next_mmr")

plt.tight_layout()
plt.show()


---
## Задание 4. Train / validation / test — **2 балла**

Разбейте данные честно: test — только для финальной проверки выбранной модели.

### Что сделать

**Шаг 1.** Отделите test (`TEST_SIZE`, `random_state=RANDOM_STATE`).

**Шаг 2.** Из оставшегося выделите validation (`VAL_SIZE`).

**Шаг 3.** Выведите размеры `X_train`, `X_val`, `X_test` (и соответствующих `y_*`).

### Подробные критерии (для проверки LLM)
- **1.0 балл:** получены train / validation / test с фиксированным `random_state`.
- **1.0 балл:** выведены размеры всех трёх частей; test не используется для выбора модели в следующих заданиях.

### Снижение баллов
- Нет отдельной validation или test → минус **1.0**.
- Размеры не выведены → минус **0.5**.


In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=VAL_SIZE, random_state=RANDOM_STATE
)

print("X_train:", X_train.shape, "X_val:", X_val.shape, "X_test:", X_test.shape)
print("y_train:", y_train.shape, "y_val:", y_val.shape, "y_test:", y_test.shape)
X_train.head()


---
## Задание 5. Baseline и метрика MAE — **3 балла**

Поставьте нижнюю планку: модель, которая всегда предсказывает среднее `next_mmr` по train.

### Что сделать

**Шаг 1.** Обучите `DummyRegressor(strategy="mean")` на train.

**Шаг 2.** Посчитайте MAE, RMSE и R2 на **validation**.

**Шаг 3.** Сохраните результат в `baseline_metrics` (словарь или строка таблицы) и явно напомните: главная метрика — `PRIMARY_METRIC` (= MAE).

### Подробные критерии (для проверки LLM)
- **1.0 балл:** baseline обучен через `DummyRegressor(strategy="mean")`.
- **1.0 балл:** MAE/RMSE/R2 посчитаны на validation.
- **1.0 балл:** результат сохранён в `baseline_metrics`; зафиксирована главная метрика MAE.

### Снижение баллов
- Метрики на train или test вместо validation → минус **1.0**.
- Нет `baseline_metrics` или пропущена MAE → минус **0.5** за каждый пропуск.


In [ ]:
dummy = DummyRegressor(strategy="mean")
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_val)

baseline_metrics = evaluate_regression("DummyRegressor(mean)", y_val, y_pred_dummy)
print("Главная метрика:", PRIMARY_METRIC)
baseline_metrics


---
## Задание 6. HistGradientBoosting и early stopping — **3 балла**

Sklearn-ориентир (не центр практики): быстрый гистограммный бустинг с автоматической остановкой.

### Что сделать

**Шаг 1.** Обучите `HistGradientBoostingRegressor` с `early_stopping=True`, умеренным `learning_rate` и достаточным `max_iter` (например 500).

**Шаг 2.** Оцените MAE/RMSE/R2 на validation; сохраните в `hgbr_metrics` (желательно с `fit_time_sec`).

**Шаг 3.** Выведите, на каком числе деревьев остановилась модель (`n_iter_`), и кратко сравните MAE с baseline.

### Подробные критерии (для проверки LLM)
- **1.0 балл:** модель обучена с `early_stopping=True`.
- **1.0 балл:** метрики на validation сохранены в `hgbr_metrics`.
- **1.0 балл:** показано `n_iter_` и сравнение MAE с baseline.

### Снижение баллов
- Early stopping выключен → минус **1.0**.
- Нет сравнения с baseline → минус **0.5**.


In [ ]:
t0 = time.perf_counter()
hgbr = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_depth=6,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=20,
    random_state=RANDOM_STATE,
)
hgbr.fit(X_train, y_train)
hgbr_time = time.perf_counter() - t0

y_pred_hgbr = hgbr.predict(X_val)
hgbr_metrics = evaluate_regression(
    "HistGradientBoosting (+early stop)", y_val, y_pred_hgbr, fit_time_sec=hgbr_time
)

print("Остановились на n_iter_ =", hgbr.n_iter_)
print(
    "MAE: HistGBM =",
    round(hgbr_metrics["MAE"], 2),
    "| baseline =",
    round(baseline_metrics["MAE"], 2),
)
hgbr_metrics


---
## Задание 7. Внешние бустинги: XGBoost, LightGBM, CatBoost — **7 баллов**

Центр практики: сравните три промышленные библиотеки на одной и той же задаче RankPulse.

### Что сделать

**Шаг 1.** Импортируйте `XGBRegressor`, `LGBMRegressor`, `CatBoostRegressor`.

**Шаг 2.** Обучите каждую модель на train. Где API позволяет — включите **early stopping** по validation (`eval_set` / callbacks). Засеките время обучения.

**Шаг 3.** Для каждой модели посчитайте MAE/RMSE/R2 на validation.

**Шаг 4.** Соберите таблицу `external_results` ровно с тремя строками внешних моделей (и при желании колонкой `fit_time_sec`).

### Подробные критерии (для проверки LLM)
- **1.5 балла:** корректно импортированы все три внешние модели.
- **3.0 балла:** обучены XGBoost, LightGBM и CatBoost без ошибок (по **1.0** за модель).
- **1.5 балла:** у ≥ 2 моделей использован early stopping / `eval_set` на validation.
- **1.0 балл:** есть `external_results` с тремя строками и метриками MAE/RMSE.

### Снижение баллов
- Отсутствует одна из трёх библиотек → минус **1.0** за каждую.
- Метрики не на validation → минус **1.5**.
- Нет early stopping ни у одной внешней модели → минус **1.0**.


In [ ]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor, early_stopping as lgb_early_stopping
from catboost import CatBoostRegressor

external_rows = []

# --- XGBoost ---
t0 = time.perf_counter()
xgb = XGBRegressor(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=RANDOM_STATE,
    objective="reg:squarederror",
    early_stopping_rounds=40,
    n_jobs=-1,
)
xgb.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)
xgb_time = time.perf_counter() - t0
xgb_pred = xgb.predict(X_val)
external_rows.append(
    evaluate_regression("XGBoost", y_val, xgb_pred, fit_time_sec=xgb_time)
)
print("XGBoost best_iteration:", getattr(xgb, "best_iteration", None))

# --- LightGBM ---
t0 = time.perf_counter()
lgbm = LGBMRegressor(
    n_estimators=800,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=RANDOM_STATE,
    verbosity=-1,
)
lgbm.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="l1",
    callbacks=[lgb_early_stopping(stopping_rounds=40, verbose=False)],
)
lgbm_time = time.perf_counter() - t0
lgbm_pred = lgbm.predict(X_val)
external_rows.append(
    evaluate_regression("LightGBM", y_val, lgbm_pred, fit_time_sec=lgbm_time)
)
print("LightGBM best_iteration_:", getattr(lgbm, "best_iteration_", None))

# --- CatBoost ---
t0 = time.perf_counter()
cbr = CatBoostRegressor(
    iterations=800,
    depth=6,
    learning_rate=0.05,
    loss_function="MAE",
    eval_metric="MAE",
    random_seed=RANDOM_STATE,
    verbose=0,
    early_stopping_rounds=40,
)
cbr.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)
cbr_time = time.perf_counter() - t0
cbr_pred = cbr.predict(X_val)
external_rows.append(
    evaluate_regression("CatBoost", y_val, cbr_pred, fit_time_sec=cbr_time)
)
print("CatBoost best_iteration_:", getattr(cbr, "best_iteration_", None))

external_results = pd.DataFrame(external_rows)
external_results


---
## Задание 8. Единый рейтинг моделей — **3 балла**

Сведите baseline, HistGBM и внешние бустинги в один рейтинг по **MAE** (меньше = лучше).

### Что сделать

**Шаг 1.** Соберите `all_results` (колонки минимум: `model`, `MAE`, `RMSE`; R2 желателен).

**Шаг 2.** Отсортируйте по MAE по возрастанию.

**Шаг 3.** Постройте bar-chart MAE по моделям (заголовок, подпись оси Y, читаемые названия моделей).

### Подробные критерии (для проверки LLM)
- **1.0 балл:** в рейтинге есть baseline, HistGBM и все три внешние модели.
- **1.0 балл:** таблица отсортирована по MAE (лучшая сверху).
- **1.0 балл:** bar-chart с заголовком и подписью оси.

### Снижение баллов
- Сортировка по другой метрике без пояснения / без MAE → минус **0.5**.
- Нет графика → минус **1.0**.


In [ ]:
all_results = pd.DataFrame(
    [
        {k: baseline_metrics[k] for k in ["model", "MAE", "RMSE", "R2"]},
        {k: hgbr_metrics[k] for k in ["model", "MAE", "RMSE", "R2"]},
        *external_results[["model", "MAE", "RMSE", "R2"]].to_dict(orient="records"),
    ]
)
all_results = all_results.sort_values("MAE", ascending=True).reset_index(drop=True)

plt.figure(figsize=(10, 4))
plt.bar(all_results["model"], all_results["MAE"], color="seagreen")
plt.title("Рейтинг моделей по MAE на validation (меньше = лучше)")
plt.ylabel("MAE (очки MMR)")
plt.xticks(rotation=20, ha="right")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

all_results


---
## Задание 9. Плюсы и минусы библиотек — **2 балла**

Соберите практическую шпаргалку для команды RankPulse: когда какую библиотеку брать.

### Что сделать

Создайте DataFrame `library_review` со строками минимум для:
`HistGradientBoosting (sklearn)`, `XGBoost`, `LightGBM`, `CatBoost`.

Колонки: `library`, `advantages`, `limitations`, `when_to_use` (тексты своими словами, по делу).

### Подробные критерии (для проверки LLM)
- **1.0 балл:** есть ≥ 4 строки по указанным библиотекам.
- **1.0 балл:** у каждой строки заполнены преимущества, ограничения и сценарий использования.

### Снижение баллов
- Пустые/копипаст-однострочники без смысла → минус **0.5**.
- Пропущена одна из ключевых библиотек → минус **0.5**.


In [ ]:
library_review = pd.DataFrame(
    [
        {
            "library": "HistGradientBoosting (sklearn)",
            "advantages": "Уже в sklearn; быстрый гистограммный бустинг; простой early stopping.",
            "limitations": "Меньше тонких настроек и экосистемы, чем у XGBoost/LightGBM/CatBoost.",
            "when_to_use": "Быстрый ориентир без новых зависимостей или учебный прототип.",
        },
        {
            "library": "XGBoost",
            "advantages": "Сильный контроль регуляризации; часто высокий потолок качества; зрелая экосистема.",
            "limitations": "Много гиперпараметров; новичкам сложнее стартовать.",
            "when_to_use": "Когда качество критично и команда готова тюнить модель.",
        },
        {
            "library": "LightGBM",
            "advantages": "Очень быстрое обучение на больших таблицах; удобен при частых переобучениях.",
            "limitations": "Leaf-wise рост легче переобучается; нужна аккуратная validation.",
            "when_to_use": "Когда важны скорость и масштаб (много игроков/признаков).",
        },
        {
            "library": "CatBoost",
            "advantages": "Устойчив на дефолтах; сильная работа с категориальными признаками (роль и др.).",
            "limitations": "Иногда медленнее LightGBM; отдельная зависимость и вес библиотеки.",
            "when_to_use": "Когда много категорий и нужен сильный результат без сложного кодирования.",
        },
    ]
)

library_review


---
## Задание 10. Финальный вывод и проверка на test — **2 балла**

Выберите лучшую модель по **validation MAE**, один раз проверьте на test и сформулируйте инженерный вывод для RankPulse.

### Что сделать

**Шаг 1.** Найдите лучшую строку в `all_results` по MAE; сравните выигрыш с baseline.

**Шаг 2.** Переобучите выбранную модель на `X_train_val` / `y_train_val` и посчитайте MAE/RMSE на **test** → `final_test_metrics`.

**Шаг 3.** В markdown напишите короткий вывод (3–5 предложений): кого берём в прод-кандидат, почему MAE важнее «красивого» R2 здесь, и что делать при большой ошибке у отдельного игрока (связь с заданием 1).

### Подробные критерии (для проверки LLM)
- **0.5 балла:** корректно выбрана лучшая модель по validation MAE.
- **0.5 балла:** модель переобучена на train+val и оценена на test (`final_test_metrics`).
- **1.0 балл:** есть связный инженерный вывод (выбор модели + метрика + правило про ручную проверку).

### Снижение баллов
- Test использовался для выбора модели → минус **1.0**.
- Нет `final_test_metrics` → минус **0.5**.
- Нет текстового вывода → минус **1.0**.


In [ ]:
best_row = all_results.iloc[0]
baseline_row = all_results.loc[all_results["model"] == "DummyRegressor(mean)"].iloc[0]
mae_gain = baseline_row["MAE"] - best_row["MAE"]

model_candidates = {
    "DummyRegressor(mean)": DummyRegressor(strategy="mean"),
    "HistGradientBoosting (+early stop)": HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=6,
        max_iter=500,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20,
        random_state=RANDOM_STATE,
    ),
    "XGBoost": XGBRegressor(
        n_estimators=800,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        objective="reg:squarederror",
        n_jobs=-1,
    ),
    "LightGBM": LGBMRegressor(
        n_estimators=int(getattr(lgbm, "best_iteration_", None) or 400),
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        verbosity=-1,
    ),
    "CatBoost": CatBoostRegressor(
        iterations=int(getattr(cbr, "best_iteration_", None) or 400),
        depth=6,
        learning_rate=0.05,
        loss_function="MAE",
        random_seed=RANDOM_STATE,
        verbose=0,
    ),
}

best_name = best_row["model"]
final_model = model_candidates[best_name]
final_model.fit(X_train_val, y_train_val)
test_pred = final_model.predict(X_test)
final_test_metrics = evaluate_regression(f"{best_name} (final test)", y_test, test_pred)

print("Лучшая по validation MAE:", best_name)
print(
    f"Validation MAE: {best_row['MAE']:.2f} | baseline MAE: {baseline_row['MAE']:.2f} | выигрыш: {mae_gain:.2f}"
)
print("Финальная проверка на test:")
print(final_test_metrics)

summary = pd.DataFrame(
    {
        "metric": [
            "best_validation_model",
            "validation_MAE",
            "baseline_validation_MAE",
            "MAE_gain_vs_baseline",
            "final_test_MAE",
            "final_test_RMSE",
        ],
        "value": [
            best_name,
            round(best_row["MAE"], 2),
            round(baseline_row["MAE"], 2),
            round(mae_gain, 2),
            round(final_test_metrics["MAE"], 2),
            round(final_test_metrics["RMSE"], 2),
        ],
    }
)
summary


**Инженерный вывод для RankPulse**

Лучшую модель выбираем по **validation MAE** (ошибка в очках MMR понятна продукту). Baseline нужен как нижняя планка: если бустинг почти не выигрывает у среднего, в матчмейкинг его рано пускать. Среди внешних библиотек обычно сравнивают XGBoost / LightGBM / CatBoost и берут победителя по validation, а **test** открывают один раз — как честную контрольную. Даже при хорошем среднем MAE игрокам с большой оценкой ошибки оставляем правило из задания 1: автопрогноз не применяем, отдаём аналитику.
